# Swing

The **ATR-prominence ZigZag swing family** — one detector (`engine/swing_detector.py`, volatility-normalized graded swings), three entry modes. Detection is shared via the `swing_zz_*` config knobs; each mode is its own strategy, look-ahead free (a swing is only visible at its `confirmation_idx`).

__The three entry modes:__
- **Flip** (`swing_flip`) — always-in *reversal*: SHORT on each confirmed swing **high**, LONG on each confirmed swing **low**, flips on the next opposite confirmation. Optional ATR trailing stop.
- **Bounce** (`swing_bounce`) — *mean-reversion*: LONG when price tests-and-rejects the latest swing **low** (SHORT the swing **high**). Swing-anchored stop + 2R target.
- **Breakout** (`swing_breakout`) — *continuation*: LONG when close breaks **above** the latest swing **high** (SHORT below the swing **low**). Swing-anchored stop + 2R target.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Flip

Always-in reversal: fade each confirmed swing pivot and flip on the next opposite confirmation.

In [ ]:
# Import swing flip strategy
from engine.strategies import SwingFlipStrategy

In [ ]:
# Backtest swing flip strategy
config = StrategyConfig()
strategy = SwingFlipStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Swing flip strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Bounce

Mean-reversion: enter on a tested-and-rejected swing level; swing-anchored stop + 2R.

In [ ]:
# Import swing bounce strategy
from engine.strategies import SwingBounceStrategy

In [ ]:
# Backtest swing bounce strategy
config = StrategyConfig()
strategy = SwingBounceStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Swing bounce strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Breakout

Continuation: enter when close breaks through the latest swing level; swing-anchored stop + 2R.

In [ ]:
# Import swing breakout strategy
from engine.strategies import SwingBreakoutStrategy

In [ ]:
# Backtest swing breakout strategy
config = StrategyConfig()
strategy = SwingBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Swing breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()